# Model Adjustment · Full Historical Evaluation

**Does the complete demand model have the lowest weekly unit error?** Evaluate every eligible historical SKU-week with an available reference price. This workbook chooses the weights. [demand_model.ipynb](demand_model.ipynb) uses the saved parameters for each decision date. This workbook reconstructs the adjustments from its fold-specific log contributions and parameters. The scores here use original weekly quotes and uncapped price response.

**Steps:** compare the original model → diagnose the calendar and price terms → replay the rolling adjustments → compare all models on identical rows. Completed-week campaign assignments depend on realized sales; interpret accuracy as retrospective scenario fit.

### Choose model weights using earlier weeks

Run this section to regenerate the parameter history and the before/after comparison. At each forecast date select the penalty and group weights from matured forecast errors only.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from contextlib import redirect_stdout
from io import StringIO

sources=["".join(cell["source"]) for cell in json.loads(Path("demand_model.ipynb").read_text())["cells"] if cell["cell_type"]=="code"]
def demand_source(marker):
    return next(source for source in sources if marker in source)

with redirect_stdout(StringIO()):
    exec(demand_source("daily_parts, coverage_parts, counts").split("display(pd.DataFrame(counts")[0])
    exec(demand_source("eda = panel.merge(pd.concat(windows)"))
    participation=pd.read_csv("synthesis data/weekly_campaign_participation.csv",parse_dates=["week_start"])
    exec(demand_source("def all_inputs(market)"))
    exec(demand_source("def raw_design(rows, spec"))
    exec(demand_source("all_market_rows={m:all_inputs(m)"))


In [ ]:
from itertools import product
from copy import deepcopy

components=["month","quarter","campaign","other_week","base_price","promo_price"]
grid=np.array(sorted(product((0.,.5,1.),repeat=6),key=lambda w:sum((1-v)**2 for v in w)))
penalties=(.1,1.,10.)
def full_blocks(weeks):
    dates=pd.DatetimeIndex(sorted(pd.unique(weeks)))
    # Weekly at launch so the first available training labels can be used immediately.
    first=dates[0]
    starts=sorted(set(dates[dates<first+pd.Timedelta(days=35)]) |
                  {g.min() for _,g in pd.Series(dates,index=dates).groupby(dates.to_period("Q"))})
    return list(zip(starts,starts[1:]+[dates[-1]+pd.Timedelta(days=7)]))

def apply_groups(fit,w):
    spec=deepcopy(fit)
    offset=1+len(fit["b_names"])
    a_weights=[w[0] if term.startswith("Fiscal month =") else w[1] if term.startswith("Fiscal quarter =")
               else w[2] if term==names[67] else w[3] for term in fit["a_names"]]
    end=offset+len(a_weights)
    spec["w"][offset:end]*=a_weights
    spec["w"][0]+=fit["neutral_a"]@(fit["w"][offset:end]-spec["w"][offset:end])
    spec["w"][end:]*=[w[5] if term==names[73] else w[4] for term in fit["p_names"]]
    spec["adjustments"]=dict(zip(components,w))
    return spec

def group_terms(frame,fit):
    _,a,p=raw_design(frame,fit)
    nb=len(fit["b_names"]); na=len(a.columns)
    a=a.fillna(fit["median"][a.columns]).clip(fit["lo"][a.columns],fit["hi"][a.columns],axis=1)
    centered=(a.to_numpy()-fit["scaler"].mean_[nb:])/fit["scaler"].scale_[nb:]-fit["neutral_a"]
    a=pd.DataFrame(centered*fit["w"][1+nb:1+nb+na],columns=a.columns)
    p=p/fit["p_scale"]*fit["w"][1+nb+na:]
    m=a.filter(like="Fiscal month =").sum(axis=1); q=a.filter(like="Fiscal quarter =").sum(axis=1)
    h=p[names[73]]
    return np.column_stack([m,q,a[names[67]],a.sum(axis=1)-m-q-a[names[67]],p.sum(axis=1)-h,h])

def training_ready(frame):
    return pd.concat([frame.week_start+pd.Timedelta(days=7),frame.last_ship+pd.Timedelta(days=1)],axis=1).max(axis=1).mask(frame.missing_ship)

def select_parameters(loss):
    pi=int(np.argmin(loss[:,0]))  # grid starts at all ones
    return pi,grid[np.argmin(loss[pi])]

pred_parts=[]; settings=[]; final_specs={}; base_specs={}; coef_rows=[]; fits_by_cutoff={}
for market,frame in all_market_rows.items():
    frame=frame.merge(settlement,on=week_key,validate="one_to_one").sort_values("week_start")
    frame["ready"]=training_ready(frame)
    blocks=full_blocks(frame.week_start)
    evaluation_dates=pd.DatetimeIndex([v[0] for v in blocks]+[frame.week_start.max()+pd.Timedelta(days=7)])
    # Delay each error contribution until its label has shipped and the week has closed.
    matured=np.zeros((len(evaluation_dates),len(penalties),len(grid)))
    losses=np.zeros_like(matured[0])
    for idx,(start,stop) in enumerate(blocks):
        losses+=matured[idx]
        pi,w=select_parameters(losses)
        future=frame.loc[frame.week_start.between(start,stop,inclusive="left")].copy()
        past=frame.loc[frame.ready.le(start)]
        obs=future[week_key+["actual_units","R","weekly_offer","promotion_scheduled","campaign_active","campaign_name","assignment_reason"]].copy()
        obs["fold_start"]=start;obs["training_rows"]=len(past)
        volume=names[3]
        obs["baseline"]=future[volume].fillna(past[volume].median() if len(past) else 0).fillna(0)/4
        if past.empty:
            obs["status"]="initialization"
            for col in ["B","BA","Q","original_full","direct_tree" ,"Q_at_95pct_R","Q_at_100pct_R","Q_at_105pct_R"]:
                obs[col]=obs.baseline
            obs["A"]=1.;obs["P"]=1.;obs["base_B"]=obs.baseline
            for gname in components:obs["log_"+gname]=0.
        else:
            obs["status"]="fitted"
            available_idx=evaluation_dates.searchsorted(future.ready.fillna(pd.Timestamp("2100-01-01")),side="left")
            with threadpool_limits(limits=2):
                for j,penalty in enumerate(penalties):
                    fit=fit_full(past,penalty,market)
                    b,a,p=predict_full(future,fit)
                    terms=group_terms(future,fit)
                    for k in np.unique(available_idx):
                        if k>=len(evaluation_dates):continue
                        mask=(available_idx==k)
                        xx=terms[mask]; yy=future.actual_units.to_numpy()[mask]; bb=b[mask]
                        for lo in range(0,len(xx),512):
                            sl=slice(lo,lo+512)
                            err=np.abs(bb[sl,None]*np.exp(xx[sl]@grid.T)-yy[sl,None])
                            matured[k,j]+=err.sum(axis=0)
                    if j==pi:
                        original=fit; adjusted=apply_groups(fit,w)
                        for gidx,gname in enumerate(components):obs["log_"+gname]=terms[:,gidx]
                        obs["base_B"]=b
                        obs["original_full"]=b*a*p
                b,a,p=predict_full(future,adjusted)
                obs["B"],obs["A"],obs["P"],obs["BA"],obs["Q"]=b,a,p,b*a,b*a*p
                for ratio in (.95,1.,1.05):
                    _,_,pr=predict_full(future,adjusted,ratio*future.R)
                    obs[f"Q_at_{round(100*ratio)}pct_R"]=b*a*pr
                def tree_matrix(d):return pd.concat(raw_design(d,original),axis=1).replace([np.inf,-np.inf],np.nan)
                tree=HistGradientBoostingRegressor(loss="poisson",max_iter=80,max_leaf_nodes=15,min_samples_leaf=50,
                        l2_regularization=2.,early_stopping=False,random_state=42)
                tm=tree_matrix(past);usable=tm.columns[tm.notna().any()]
                if len(past)<2*tree.min_samples_leaf or not len(usable):
                    obs["direct_tree"]=past.actual_units.mean()  # unsplit Poisson tree
                else:
                    tree.fit(tm[usable],past.actual_units);obs["direct_tree"]=tree.predict(tree_matrix(future)[usable])
                fits_by_cutoff[market,start]=adjusted
        pred_parts.append(obs)
        settings.append(dict(country=market,start=start,stop=stop,training_rows=len(past),rows=len(future),
                             penalty=penalties[pi],**dict(zip(components,w))))
    # Refit for future decisions, independently of the out-of-time accuracy table.
    losses+=matured[-1]
    pi,w=select_parameters(losses)
    deployment_date=evaluation_dates[-1]
    ready=frame.loc[frame.ready.le(deployment_date)]
    with threadpool_limits(limits=2):
        base_specs[market]=fit_full(ready,penalties[pi],market)
    spec=apply_groups(base_specs[market],w);final_specs[market]=spec
    for term,coef in zip(spec["b_names"]+spec["a_names"]+spec["p_names"],spec["w"][1:]):
        coef_rows.append(dict(market=market,term=term,standardized_log_coefficient=coef))
    settings.append(dict(country=market,start=deployment_date,stop=pd.NaT,training_rows=len(ready),rows=0,
                         penalty=penalties[pi],**dict(zip(components,w))))
    print(market,"complete",len(frame),"rows",flush=True)
full_predictions=pd.concat(pred_parts,ignore_index=True)
assert not full_predictions.duplicated(week_key).any()
assert len(full_predictions)==sum(map(len,all_market_rows.values()))
columns={"baseline":"28-day run rate","B":"B","BA":"B times A","Q":"B × A × P(p)","direct_tree":"direct tree",
         "original_full":"Original full"}
def summarize_full(frame,groups):
    values=[]
    for key,d in frame.groupby(groups):
        key=key if isinstance(key,tuple) else (key,)
        for col,label in columns.items():
            error=d[col]-d.actual_units
            assert np.isfinite(error).all()
            values.append(dict(zip(groups,key),model=label,rows=len(d),MAE=error.abs().mean(),
                               bias=error.mean(),WAPE=error.abs().sum()/d.actual_units.sum()))
    return pd.DataFrame(values)
full_scores=summarize_full(full_predictions,["country"])
full_predictions.to_csv(out_dir/"full_history_adjustment_predictions.csv",index=False)
full_scores.to_csv(out_dir/"full_history_adjustment_scores.csv",index=False)
pd.DataFrame(settings).to_csv(out_dir/"full_history_demand_parameters.csv",index=False)
print(full_predictions.groupby(["country","status"]).size().to_string())
print(full_scores.pivot(index="country",columns="model",values="MAE").round(3).to_string())

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

folder=Path("pricewise_outputs")
data=pd.read_csv(folder/"full_history_adjustment_predictions.csv",parse_dates=["week_start","fold_start"])
params=pd.read_csv(folder/"full_history_demand_parameters.csv",parse_dates=["start","stop"])
coverage=pd.read_csv(folder/"full_history_demand_coverage.csv")
key=["country","product_id","segment","product_type","week_start"]
assert not data.duplicated(key).any()
assert data.groupby("country").size().to_dict()==coverage.set_index("market").scored.to_dict()
print(coverage.to_string(index=False))

market  panel_rows  panel_weeks  unavailable_label_or_coverage  eligible_missing_R  scored  scored_weeks  initialization first_scored last_scored
    US      345399          178                         282626                2322   60451           155              97   2023-04-03  2026-08-17
    CA      288150          178                         247623                3041   37486           155              40   2023-04-03  2026-08-17


### 1. Reproduce the original structure
B forecasts units at R. A applies fiscal month, quarter, campaign and other weekly effects. P applies the base price response and promotion–price interaction. Within each market and cutoff, reconstruct the original forecast by giving every component weight one. The first available decision weeks use the documented starting forecast until training labels have matured.

In [2]:
groups=["month","quarter","campaign","other_week","base_price","promo_price"]
logs=data[["log_"+g for g in groups]].to_numpy()
target=data.actual_units.to_numpy()
base=data.base_B.to_numpy()
def forecast(multipliers):
    return base*np.exp(np.sum(logs*multipliers,axis=1))
def market_mae(values):
    return pd.Series(np.abs(values-target)).groupby(data.country).mean()
original=forecast(np.ones(6))
assert np.allclose(original,data.original_full)
print(pd.DataFrame({label:market_mae(pred) for label,pred in {
    "28-day run rate":data.baseline,"B":base,"B times A":forecast([1,1,1,1,0,0]),
    "Original full":original,"direct tree":data.direct_tree}.items()}).round(3).to_string())

         28-day run rate       B  B times A  Original full  direct tree
country                                                                
CA                 6.383   6.217      6.144         11.093        5.792
US                20.781  21.252     20.423         20.459       19.441


### 2. Locate the error
Summarize every calendar month observed across years and both campaign states. The table shows how the original model's absolute error changes when adding A and P. Positive values mean error increased. The ablation table neutralizes one contribution at a time across the complete sample. This holds the fitted coefficients fixed.

In [3]:
original_BA=forecast([1,1,1,1,0,0])
diag=data[["country","week_start","campaign_active"]].copy()
diag["month"]=diag.week_start.dt.month
diag["delta_A"]=abs(original_BA-target)-abs(base-target)
diag["delta_P"]=abs(original-target)-abs(original_BA-target)
for group in ("month","campaign_active"):
    print(diag.groupby(["country",group]).agg(rows=("delta_A","size"),A_MAE_change=("delta_A","mean"),
          P_MAE_change=("delta_P","mean")).round(3).to_string())
ablate={}
for i,term in enumerate(groups):
    w=np.ones(6);w[i]=0
    ablate[term]=market_mae(forecast(w))-market_mae(original)
print("MAE change after neutralizing each term")
print(pd.DataFrame(ablate).round(3).to_string())

               rows  A_MAE_change  P_MAE_change
country month                                  
CA      1      2255         0.444         0.038
        2      2020         0.288         0.008
        3      3017         0.217        -0.006
        4      3402         0.207        -0.008
        5      4121         0.009         0.003
        6      4188         0.038         0.006
        7      4165         0.206         0.006
        8      3875        -0.172        47.925
        9      3473        -0.078         0.018
        10     2331         0.017         0.008
        11     2303        -1.484        -0.119
        12     2336        -1.033        -0.053
US      1      3660         0.877         0.121
        2      3224         0.718        -0.003
        3      4891         0.889        -0.057
        4      5794        -0.001         0.042
        5      6630         0.324         0.010
        6      6699        -0.261        -0.004
        7      6977        -0.151       

### 3. Replay the adjustment selected at each cutoff
Demand refits at launch and then quarterly. It selects the regularization penalty using all earlier, label-available forecasts and compares 729 combinations of six multipliers at the selected penalty. A multiplier of 0 neutralizes a group; 0.5 halves its log contribution. Each scored row joins only to the decision that produced it. Final deployment settings, fitted after scoring, appear separately.

In [4]:
folds=params.loc[params.rows.gt(0)]
aligned=data[["country","fold_start"]].merge(folds,left_on=["country","fold_start"],right_on=["country","start"],
    validate="many_to_one",how="left",sort=False)
assert aligned[groups].notna().all().all()
w=aligned[groups].to_numpy()
adjusted=forecast(w)
wa=w.copy();wa[:,4:]=0
adjusted_BA=forecast(wa)
assert np.allclose(adjusted,data.Q) and np.allclose(adjusted_BA,data.BA)
print(folds[["country","start","training_rows","rows","penalty",*groups]].to_string(index=False))
print("Final full-history fit for future decisions")
print(params.loc[params.rows.eq(0),["country","start","training_rows","penalty",*groups]].to_string(index=False))

country      start  training_rows  rows  penalty  month  quarter  campaign  other_week  base_price  promo_price
     US 2023-04-03              0    97      0.1    1.0      1.0       1.0         1.0         1.0          1.0
     US 2023-04-10             57   270      0.1    1.0      1.0       1.0         1.0         1.0          1.0
     US 2023-04-17            248   340     10.0    1.0      1.0       1.0         1.0         1.0          1.0
     US 2023-04-24            527   401      1.0    1.0      1.0       1.0         0.0         1.0          1.0
     US 2023-05-01            845  3660      1.0    1.0      1.0       1.0         0.0         1.0          1.0
     US 2023-07-03           4413  5237     10.0    1.0      1.0       1.0         0.5         0.5          1.0
     US 2023-10-02           9672  5445      0.1    1.0      1.0       1.0         1.0         0.0          1.0
     US 2024-01-01          15060  5267      0.1    0.0      0.0       1.0         1.0         0.0      

### 4. Score every forecast and check the extreme case
The primary result includes every eligible record. Supplement it with initialization counts, model-only results, campaign participation and the result excluding the previously investigated Canadian invoice. These secondary comparisons use the same forecasts as the primary result.

In [5]:
comparison=data[["country","product_id","week_start","actual_units","status","campaign_active"]].copy()
for label,pred in {"28-day run rate":data.baseline,"B":base,"B times A":adjusted_BA,
                   "Adjusted full":adjusted,"direct tree":data.direct_tree,"Original full":original}.items():
    comparison[label]=abs(pred-target)
cols=comparison.columns[6:].tolist()
summary=comparison.groupby("country")[cols].mean()
print("Full eligible history");print(summary.round(3).to_string())
print("By forecast status");print(comparison.groupby(["country","status"])[cols].mean().round(3).to_string())
print("By participation");print(comparison.groupby(["country","campaign_active"])[cols].mean().round(3).to_string())
case=comparison.country.eq("CA") & comparison.product_id.eq("66AD491QON") & comparison.week_start.eq("2026-08-17")
assert case.sum()==1
print("Sensitivity: discovery observation excluded")
print(comparison.loc[~case].groupby("country")[cols].mean().round(3).to_string())

Full eligible history
         28-day run rate       B  B times A  Adjusted full  direct tree  Original full
country                                                                               
CA                 6.383   6.217      6.075         10.244        5.792         11.093
US                20.781  21.252     20.428         20.448       19.441         20.459
By forecast status
                        28-day run rate       B  B times A  Adjusted full  direct tree  Original full
country status                                                                                       
CA      fitted                    6.382   6.217      6.075         10.248        5.791         11.098
        initialization            6.688   6.688      6.688          6.688        6.688          6.688
US      fitted                   20.785  21.258     20.432         20.453       19.444         20.463
        initialization           17.889  17.889     17.889         17.889       17.889         17.889

### Decision

**CA.** Adjusted full MAE 10.244; direct tree 5.792. The strict lowest-MAE target is not met on the full history.

**US.** Adjusted full MAE 20.448; direct tree 19.441. The strict lowest-MAE target is not met on the full history.

Keep the original and adjusted price curves for scenario comparisons. The Canadian full-history result is sensitive to the August 17 quote: the price coefficient and group strengths for that row were learned using only data available before its fold. Its error is retained above.

**Next investigation.** Compare B × A with B × A × P(p) on the same rows and inspect the largest increases in absolute error. The dedicated guardrail section in [demand_model.ipynb](demand_model.ipynb) follows that diagnosis.